# HNL mu-pi: frames -> binned sensitivity (cafpybara)

Mechanical pipeline, not a result: the selection, the variable and the statistics are placeholders.

1. Load the BNB+CORSIKA MC and the overlaid HNL frames, apply `ana.ALL_CUTS`.
2. Histogram the angle of the longest track to z (`long1.theta_z`) in a few bins into hnlpi0's pyhf-input dicts. The BNB dict gets
   the full detvar+norm covariance (`syst_cov`) from `get_total_cov`; the HNL dicts carry MC statistics only.
3. `upper_limit_mu` keeps the bin-to-bin correlations of the background (covariance modes as `histosys` nuisances).
4. Expected |U|^2 limit per mass.

Not in the background yet: off-beam (needs the on-beam gate count from the data frames being rebuilt), multisim (`rate`) and cosmic systematics.

In [ ]:
import sys
from pathlib import Path

sys.path[:0] = ["/exp/sbnd/app/users/sdas1/cafpyana", "/exp/sbnd/app/users/sdas1"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import cafpybara.analyses.hnl_mupi as ana
from cafpybara.analyses.hnl_mupi import config

STREAM = True            # read the frames from dCache over xrootd (no /pnfs mount, e.g. on JupyterHub)
MAX_EVENTS = None        # layout-check switch: first N events of every frame
IF_SAVE = False          # write the pyhf-input pickles (scratch is write-once: bump SUFFIX)
MASSES = (300, 365)
CUTS = ana.ALL_CUTS
c = lambda *p: tuple(list(p) + [""] * (6 - len(p)))
VAR = c("long1", "theta_z")                    # placeholder: angle of the longest track to z [deg]
BINS = np.linspace(0, 180, 7)
VAR_LABEL = "Angle of longest track to z [deg]"
TARGET = config.TARGET_POT
SUFFIX = "theta_z_v1"
OUT_DIR = Path.home() / "scratch" / f"hnl_mupi_sensitivity_{SUFFIX}"
PLOT_DIR = Path("/exp/sbnd/app/users/sdas1/hnl/plots/sensitivity_pipeline")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

## 1. Load and select

In [ ]:
if STREAM:
    ana.ensure_token()
    url = ana.xrootd_url
else:
    url = lambda p: p
bnb0, POT_BNB, _ = ana.load_mc(url(config.BNB_FILE), max_events=MAX_EVENTS)
hnl0, POT_HNL, HNL_INFO = {}, {}, {}
for m in MASSES:
    hnl0[m], POT_HNL[m], HNL_INFO[m] = ana.load_mchnl(url(config.HNL_FILES[m]), max_events=MAX_EVENTS)

inside = lambda d: d[(d[VAR] >= BINS[0]) & (d[VAR] <= BINS[-1])]       # rows the histogram keeps
bnb = inside(ana.select(bnb0, CUTS))
hnl = {m: inside(ana.select(h, CUTS)) for m, h in hnl0.items()}
hnl_sig = {m: h[h.signal == ana.signal_dict["hnl"]] for m, h in hnl.items()}
print(f"BNB: {len(bnb)} of {len(bnb0)} slices pass, POT = {POT_BNB:.3e}")
for m in MASSES:
    print(f"m{m}: {len(hnl_sig[m])} HNL-matched of {len(hnl0[m])} slices pass, POT = {POT_HNL[m]:.3e}, |U|^2 = {HNL_INFO[m]['simU']:.1e}")

## 2. pyhf inputs

Everything is scaled to `config.TARGET_POT`. The background errors come from `get_total_cov` (`detv` + `norm`); the signal dicts have MC stat only.

In [ ]:
bkg_dict = ana.pyhf_input_dict(bnb, TARGET, VAR, BINS, scale=TARGET / POT_BNB, weighted=True,
                               systs=ana.systs_input(POT_BNB, cuts=CUTS))
sig_dict = {m: ana.pyhf_input_dict(hnl_sig[m], TARGET, VAR, BINS, scale=TARGET / POT_HNL[m], weighted=True,
                                   m=m, U=HNL_INFO[m]["simU"]) for m in MASSES}
display(pd.DataFrame({"BNB": bkg_dict["counts"], "BNB MC err": bkg_dict["mcstat_err"], "BNB syst err": bkg_dict["syst_err"],
                      **{f"m{m} (|U|^2={HNL_INFO[m]['simU']:.0e})": sig_dict[m]["counts"] for m in MASSES}},
                     index=pd.IntervalIndex.from_breaks(BINS)).round(4))

In [ ]:
cov = bkg_dict["syst_cov"]
corr = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)) + 1e-300)
print("covariance modes kept:", len(ana.cov_modes(cov)), "of", len(cov))
fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 4.5))
x = 0.5 * (BINS[1:] + BINS[:-1]); w = np.diff(BINS)
a0.bar(x, bkg_dict["counts"], width=w, color="#cbd5e1", edgecolor="k", label="BNB + cosmics MC")
a0.errorbar(x, bkg_dict["counts"], bkg_dict["total_err"], fmt="none", color="k", capsize=3, label="MC stat $\\oplus$ syst (diag)")
for m in MASSES:
    k = ana.HNL_DISPLAY_SCALE[m]
    a0.step(BINS, np.r_[sig_dict[m]["counts"], sig_dict[m]["counts"][-1]] * k, where="post", label=ana.hnl_label(m, HNL_INFO[m]["simU"], k))
a0.set(xlabel=VAR_LABEL, ylabel=f"slices / {TARGET:.0e} POT"); a0.legend(fontsize=8)
im = a1.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r", origin="lower", extent=[BINS[0], BINS[-1]] * 2)
a1.set(xlabel=VAR_LABEL, ylabel=VAR_LABEL, title="BNB systematic correlation"); fig.colorbar(im, ax=a1)
fig.tight_layout(); fig.savefig(PLOT_DIR / f"inputs_{SUFFIX}.pdf"); plt.show()

## 3. Expected limits

In [ ]:
inputs = {"mcbnb": bkg_dict, **sig_dict}
tab = ana.sensitivity_table(inputs, mc_stat=(True, False))
display(tab.set_index(["mass [MeV]", "mc_stat"]).filter(["S", "B", "S/sqrt(B)", "mu95", "|U|^2 limit"]))
fig, ax = plt.subplots(figsize=(6, 4))
t = tab[tab.mc_stat]
ax.errorbar(t["mass [MeV]"], t["|U|^2 limit"], [t["|U|^2 limit"] - t["|U|^2 (-1s)"], t["|U|^2 (+1s)"] - t["|U|^2 limit"]], fmt="o-", capsize=3)
ax.set(xlabel="HNL mass [MeV]", ylabel="$|U_{\\mu 4}|^2$ 95% CL (expected)", yscale="log"); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(PLOT_DIR / f"limit_{SUFFIX}.pdf"); plt.show()

## 4. Save

In [ ]:
if IF_SAVE:
    print(ana.save_pyhf_inputs(str(OUT_DIR), SUFFIX, bkg_dict, sig_dict))